# 02 — Feature Engineering Benner: Processos Jurídicos

Este notebook continua a partir do notebook:

```text
01_eda_forense_benner_processos.ipynb
```

Ele transforma a base analítica da EDA em uma **ABT modelável** para modelos de jurimetria e priorização financeira.

## Objetivos

Criar features para possíveis modelos como:

1. Previsão de `Estimativa_De_Perda`.
2. Priorização financeira de processos.
3. Identificação de processos com maior risco de perda.
4. Recomendação de estratégia de acordo.
5. Segmentação de causas de massa vs causas estratégicas.
6. Monitoramento de processos sem andamento.
7. Score de exposição jurídica.

## Arquivo de entrada esperado

```text
data/processed/benner_processos_eda_analitica.parquet
```

Esse arquivo é gerado no primeiro notebook.

## 1. Imports e configuração

In [ ]:
import pandas as pd
import numpy as np
import re
import unicodedata
from pathlib import Path

pd.set_option("display.max_columns", 250)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

BASE_DIR = Path("..")  # altere para Path('.') se o notebook estiver na raiz
PROCESSED_DIR = BASE_DIR / "data" / "processed"
REPORTS_DIR = BASE_DIR / "outputs" / "reports"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

INPUT_FILE = PROCESSED_DIR / "benner_processos_eda_analitica.parquet"

TECH_ID_COL = "OGC_FID"
INTERNAL_PROCESS_ID_COL = "Identificador"
JUDICIAL_PROCESS_ID_COL = "Numerodistribuicao"
SITUATION_COL = "Situacao"
MAIN_DATE_COL = "Data"
START_DATE_COL = "Datainicial"
END_DATE_COL = "Dataencerramento"
PORTFOLIO_COL = "Carteira"
AUTHOR_DOC_COL = "Cpf_Autor"
AUTHOR_COL = "Autor"
EXTERNAL_LAWYER_COL = "Credenciado"
PRODUCT_COL = "Nm_Produto"
INTERNAL_LAWYER_COL = "Adv_Interno"
COMPANY_COL = "Nm_Empresa"
CLOSING_REASON_COL = "Motivo_Encerramento"
COURT_UNIT_COL = "Denominacao"
ACTION_COL = "Nm_Acao"
SUBJECT_COL = "Nm_Assunto"
VALUE_COL = "Valorajuizado"
RELEVANT_PROCESS_COL = "Processorelevante"
SERVICE_AREA_COL = "Localdeservico"
OLD_PROCESS_COL = "Nrprocessoantigo"
UF_COL = "Uf"
SUB_REASON_COL = "Sub_Motivo"
DISTRICT_COL = "Comarca"
DAYS_SINCE_LAST_MOVE_COL = "Qtddias_Ultimoandamento"
PROCESS_TYPE_COL = "Tipoprocesso"
SECONDARY_DATE_COL = "Dt_Cad2"
CONTRACT_NUMBER_COL = "Numerodocontrato"
CONTRACT_DATE_COL = "Datadocontrato"
SETTLEMENT_ELIGIBLE_COL = "Passiveldeacordo"
OPPOSING_LAWYER_COL = "Advogadoadverso"
PROCESS_PHASE_COL = "Fasedoprocesso"
LOSS_ESTIMATE_COL = "Estimativa_De_Perda"

print(INPUT_FILE)

## 2. Funções utilitárias

In [ ]:
NULL_STRINGS = {"", "null", "nan", "none", "na", "n/a", "-", "não informado", "nao informado", "sem informação", "sem informacao"}

def normalize_text(text):
    if pd.isna(text):
        return None
    text = str(text).strip().lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(c for c in text if not unicodedata.combining(c))
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    if text in NULL_STRINGS:
        return None
    return text

def existing_cols(df, cols):
    return [c for c in cols if c in df.columns]

def save_report(df, filename):
    path = REPORTS_DIR / filename
    df.to_csv(path, index=False)
    print(f"Salvo: {path}")

def sanitize_col_name(text):
    text = normalize_text(text)
    if text is None:
        return "nao_informado"
    text = re.sub(r"[^a-zA-Z0-9_]", "_", text)
    text = re.sub(r"_+", "_", text).strip("_")
    return text

def safe_divide(a, b):
    return np.where((b == 0) | pd.isna(b), np.nan, a / b)

def parse_yes_no(value):
    t = normalize_text(value)
    if t in ["sim", "s", "yes", "y", "true", "1"]:
        return 1
    if t in ["nao", "n", "no", "false", "0"]:
        return 0
    return np.nan

def map_loss_estimate(value):
    t = normalize_text(value)
    if t is None:
        return np.nan
    if "remoto" in t:
        return 0
    if "possivel" in t:
        return 1
    if "provavel" in t:
        return 2
    return np.nan

def loss_estimate_weight(value):
    t = normalize_text(value)
    if t is None:
        return 0.30
    if "remoto" in t:
        return 0.10
    if "possivel" in t:
        return 0.50
    if "provavel" in t:
        return 0.90
    return 0.30

## 3. Carregar base analítica da EDA

In [ ]:
df = pd.read_parquet(INPUT_FILE)

print(df.shape)
df.head()

## 4. Definir objetivos possíveis e targets

Nesta base, existem vários objetivos possíveis.

## Objetivo A — Prever estimativa de perda

Target:

```text
Estimativa_De_Perda
```

Pode ser tratado como:

- classificação ordinal: Remoto < Possível < Provável;
- classificação binária: Provável vs demais;
- classificação binária: Possível/Provável vs Remoto.

## Objetivo B — Priorização financeira

Target/score proxy:

```text
exposicao_risco_proxy = valor_ajuizado × peso_estimativa_perda
```

Esse é útil para ranking e priorização.

## Objetivo C — Passível de acordo

Target:

```text
Passiveldeacordo
```

Pode ajudar a recomendar quais processos entram em régua de negociação.

## Objetivo D — Processo relevante

Target:

```text
Processorelevante
```

Pode ajudar a identificar casos que deveriam receber governança especial.

Neste notebook vamos preparar features para todos esses cenários, mas sem treinar modelo ainda.

In [ ]:
df_fe = df.copy()

# Garantir campos financeiros e targets auxiliares
if "valor_ajuizado" not in df_fe.columns and VALUE_COL in df_fe.columns:
    df_fe[VALUE_COL] = pd.to_numeric(df_fe[VALUE_COL], errors="coerce")
    df_fe["valor_ajuizado"] = df_fe[VALUE_COL]

if LOSS_ESTIMATE_COL in df_fe.columns:
    df_fe["estimativa_perda_score"] = df_fe[LOSS_ESTIMATE_COL].apply(map_loss_estimate)
    df_fe["target_perda_provavel"] = (df_fe["estimativa_perda_score"] == 2).astype("float")
    df_fe.loc[df_fe["estimativa_perda_score"].isna(), "target_perda_provavel"] = np.nan

    df_fe["target_perda_possivel_ou_provavel"] = (df_fe["estimativa_perda_score"] >= 1).astype("float")
    df_fe.loc[df_fe["estimativa_perda_score"].isna(), "target_perda_possivel_ou_provavel"] = np.nan

    df_fe["peso_estimativa_perda"] = df_fe[LOSS_ESTIMATE_COL].apply(loss_estimate_weight)
else:
    df_fe["peso_estimativa_perda"] = 0.30

df_fe["exposicao_risco_proxy"] = df_fe["valor_ajuizado"].fillna(0) * df_fe["peso_estimativa_perda"]

if SETTLEMENT_ELIGIBLE_COL in df_fe.columns:
    df_fe["target_passivel_acordo"] = df_fe[SETTLEMENT_ELIGIBLE_COL].apply(parse_yes_no)

if RELEVANT_PROCESS_COL in df_fe.columns:
    df_fe["target_processo_relevante"] = df_fe[RELEVANT_PROCESS_COL].apply(parse_yes_no)

target_summary = pd.DataFrame({
    "target": [
        "target_perda_provavel",
        "target_perda_possivel_ou_provavel",
        "target_passivel_acordo",
        "target_processo_relevante",
    ],
    "existe": [
        "target_perda_provavel" in df_fe.columns,
        "target_perda_possivel_ou_provavel" in df_fe.columns,
        "target_passivel_acordo" in df_fe.columns,
        "target_processo_relevante" in df_fe.columns,
    ],
    "qtd_nao_nulos": [
        df_fe["target_perda_provavel"].notna().sum() if "target_perda_provavel" in df_fe.columns else 0,
        df_fe["target_perda_possivel_ou_provavel"].notna().sum() if "target_perda_possivel_ou_provavel" in df_fe.columns else 0,
        df_fe["target_passivel_acordo"].notna().sum() if "target_passivel_acordo" in df_fe.columns else 0,
        df_fe["target_processo_relevante"].notna().sum() if "target_processo_relevante" in df_fe.columns else 0,
    ]
})
save_report(target_summary, "target_summary_feature_engineering_benner.csv")
target_summary

## 5. Features financeiras

Criamos features sobre valor ajuizado:

- valor nulo;
- valor zero/negativo;
- log do valor;
- faixas de valor;
- percentil do valor;
- indicadores de alto valor.

Essas features ajudam tanto modelos quanto regras de priorização.

In [ ]:
df_fe["valor_ajuizado"] = pd.to_numeric(df_fe["valor_ajuizado"], errors="coerce")

df_fe["valor_ajuizado_is_null"] = df_fe["valor_ajuizado"].isna().astype(int)
df_fe["valor_ajuizado_zero_ou_negativo"] = (df_fe["valor_ajuizado"].fillna(0) <= 0).astype(int)
df_fe["valor_ajuizado_log1p"] = np.where(
    df_fe["valor_ajuizado"].fillna(0) > 0,
    np.log1p(df_fe["valor_ajuizado"]),
    0
)

# Faixas por regra de negócio
bins = [-np.inf,0,1000,5000,10000,25000,50000,100000,250000,500000,1000000,np.inf]
labels = ["00_sem_valor_ou_zero","01_ate_1k","02_1k_5k","03_5k_10k","04_10k_25k","05_25k_50k","06_50k_100k","07_100k_250k","08_250k_500k","09_500k_1mm","10_acima_1mm"]

df_fe["faixa_valor_ajuizado"] = pd.cut(
    df_fe["valor_ajuizado"].fillna(0),
    bins=bins,
    labels=labels,
    include_lowest=True
).astype(str)

# Percentil e flags de alto valor
df_fe["valor_ajuizado_percentil"] = df_fe["valor_ajuizado"].rank(pct=True)

df_fe["flag_top_1pct_valor"] = (df_fe["valor_ajuizado_percentil"] >= 0.99).astype(int)
df_fe["flag_top_5pct_valor"] = (df_fe["valor_ajuizado_percentil"] >= 0.95).astype(int)
df_fe["flag_top_10pct_valor"] = (df_fe["valor_ajuizado_percentil"] >= 0.90).astype(int)

financial_features = [
    "valor_ajuizado",
    "valor_ajuizado_is_null",
    "valor_ajuizado_zero_ou_negativo",
    "valor_ajuizado_log1p",
    "faixa_valor_ajuizado",
    "valor_ajuizado_percentil",
    "flag_top_1pct_valor",
    "flag_top_5pct_valor",
    "flag_top_10pct_valor",
    "exposicao_risco_proxy",
]
df_fe[financial_features].head()

In [ ]:
financial_features_report = df_fe[financial_features].describe(include="all").T
financial_features_report.to_csv(REPORTS_DIR / "financial_features_report_benner.csv")
financial_features_report

## 6. Features temporais e de ciclo de vida

Criamos features de:

- ano/mês de início;
- idade do processo;
- duração até encerramento;
- tempo desde último andamento;
- flags de inatividade;
- processo encerrado/em curso.

Cuidado: `Dataencerramento` e `Situacao` podem ser leakage se o modelo for prever risco antes do encerramento.

In [ ]:
date_cols = [c for c in [MAIN_DATE_COL, START_DATE_COL, END_DATE_COL, SECONDARY_DATE_COL, CONTRACT_DATE_COL] if c in df_fe.columns]
for col in date_cols:
    df_fe[col] = pd.to_datetime(df_fe[col], errors="coerce")

candidate_ref_dates = []
for col in [MAIN_DATE_COL, END_DATE_COL, SECONDARY_DATE_COL]:
    if col in df_fe.columns:
        mx = df_fe[col].max()
        if pd.notna(mx):
            candidate_ref_dates.append(mx)
DATA_REF = max(candidate_ref_dates) if candidate_ref_dates else pd.Timestamp.today().normalize()

for col in date_cols:
    df_fe[f"{col}_ano"] = df_fe[col].dt.year
    df_fe[f"{col}_mes"] = df_fe[col].dt.month
    df_fe[f"{col}_trimestre"] = df_fe[col].dt.quarter
    df_fe[f"{col}_ano_mes"] = df_fe[col].dt.to_period("M").astype(str)
    df_fe[f"{col}_is_null"] = df_fe[col].isna().astype(int)

if START_DATE_COL in df_fe.columns:
    df_fe["idade_processo_dias"] = (DATA_REF - df_fe[START_DATE_COL]).dt.days
    df_fe["idade_processo_anos"] = df_fe["idade_processo_dias"] / 365.25
    df_fe["idade_processo_log1p"] = np.log1p(df_fe["idade_processo_dias"].clip(lower=0).fillna(0))

if END_DATE_COL in df_fe.columns and START_DATE_COL in df_fe.columns:
    df_fe["duracao_ate_encerramento_dias"] = (df_fe[END_DATE_COL] - df_fe[START_DATE_COL]).dt.days
    df_fe["duracao_ate_encerramento_log1p"] = np.log1p(df_fe["duracao_ate_encerramento_dias"].clip(lower=0).fillna(0))

if DAYS_SINCE_LAST_MOVE_COL in df_fe.columns:
    df_fe[DAYS_SINCE_LAST_MOVE_COL] = pd.to_numeric(df_fe[DAYS_SINCE_LAST_MOVE_COL], errors="coerce")
    df_fe["dias_ultimo_andamento_log1p"] = np.log1p(df_fe[DAYS_SINCE_LAST_MOVE_COL].clip(lower=0).fillna(0))
    df_fe["flag_sem_andamento_30d"] = (df_fe[DAYS_SINCE_LAST_MOVE_COL] >= 30).astype(int)
    df_fe["flag_sem_andamento_90d"] = (df_fe[DAYS_SINCE_LAST_MOVE_COL] >= 90).astype(int)
    df_fe["flag_sem_andamento_180d"] = (df_fe[DAYS_SINCE_LAST_MOVE_COL] >= 180).astype(int)
    df_fe["flag_sem_andamento_360d"] = (df_fe[DAYS_SINCE_LAST_MOVE_COL] >= 360).astype(int)

if SITUATION_COL in df_fe.columns:
    df_fe["situacao_norm"] = df_fe[SITUATION_COL].apply(normalize_text)
    df_fe["flag_encerrado"] = df_fe["situacao_norm"].apply(lambda x: 1 if x and "encerr" in x else 0)
    df_fe["flag_em_curso"] = df_fe["situacao_norm"].apply(lambda x: 1 if x and ("curso" in x or "aberto" in x) else 0)

temporal_cols = [c for c in df_fe.columns if any(k in c.lower() for k in ["idade_processo", "duracao_ate", "ultimo_andamento", "sem_andamento", "flag_encerrado", "flag_em_curso"])]
df_fe[temporal_cols].head()

## 7. Features de contrato

A coluna `Numerodocontrato` pode conter um ou múltiplos contratos associados ao processo.

Criamos:
- flag de contrato informado;
- quantidade aproximada de contratos;
- data do contrato;
- idade do contrato;
- diferença entre contrato e início do processo.

In [ ]:
if CONTRACT_NUMBER_COL in df_fe.columns:
    def count_contracts(value):
        if pd.isna(value):
            return 0
        text = str(value)
        # separadores comuns
        parts = re.split(r"[;,|/\s]+", text)
        parts = [p for p in parts if p.strip()]
        return len(set(parts))

    df_fe["flag_tem_contrato"] = df_fe[CONTRACT_NUMBER_COL].notna().astype(int)
    df_fe["qtd_contratos_aprox"] = df_fe[CONTRACT_NUMBER_COL].apply(count_contracts)
    df_fe["qtd_contratos_log1p"] = np.log1p(df_fe["qtd_contratos_aprox"])

if CONTRACT_DATE_COL in df_fe.columns:
    df_fe[CONTRACT_DATE_COL] = pd.to_datetime(df_fe[CONTRACT_DATE_COL], errors="coerce")
    df_fe["flag_data_contrato_nula"] = df_fe[CONTRACT_DATE_COL].isna().astype(int)
    df_fe["idade_contrato_dias"] = (DATA_REF - df_fe[CONTRACT_DATE_COL]).dt.days
    df_fe["idade_contrato_log1p"] = np.log1p(df_fe["idade_contrato_dias"].clip(lower=0).fillna(0))

    if START_DATE_COL in df_fe.columns:
        df_fe["dias_contrato_ate_processo"] = (df_fe[START_DATE_COL] - df_fe[CONTRACT_DATE_COL]).dt.days
        df_fe["dias_contrato_ate_processo_log1p"] = np.log1p(df_fe["dias_contrato_ate_processo"].clip(lower=0).fillna(0))

contract_cols = [c for c in df_fe.columns if "contrato" in c.lower()]
df_fe[contract_cols].head()

## 8. Features de autor, advogados e escritórios

Campos de pessoas/entidades geralmente têm alta cardinalidade. Em vez de one-hot puro, criamos:

- flag de preenchimento;
- frequência histórica da entidade;
- top N + outros;
- comprimento/número de tokens do nome.

Isso ajuda a capturar litigância recorrente sem explodir dimensionalidade.

In [ ]:
entity_cols = [c for c in [
    AUTHOR_COL, AUTHOR_DOC_COL, EXTERNAL_LAWYER_COL, INTERNAL_LAWYER_COL,
    OPPOSING_LAWYER_COL, COMPANY_COL, SERVICE_AREA_COL, COURT_UNIT_COL
] if c in df_fe.columns]

def add_entity_basic_features(df, col, top_n=50):
    df = df.copy()
    norm_col = f"{col}_norm"
    df[norm_col] = df[col].apply(normalize_text)

    df[f"{col}_is_null"] = df[norm_col].isna().astype(int)
    df[f"{col}_num_tokens"] = df[norm_col].apply(lambda x: len(x.split()) if isinstance(x, str) else 0)

    freq = df[norm_col].value_counts(dropna=True).to_dict()
    df[f"{col}_freq"] = df[norm_col].map(freq).fillna(0)
    df[f"{col}_freq_log1p"] = np.log1p(df[f"{col}_freq"])

    top_values = set(df[norm_col].value_counts(dropna=True).head(top_n).index)
    df[f"{col}_top{top_n}_ou_outros"] = df[norm_col].apply(lambda x: x if x in top_values else "outros_ou_nulo")

    return df

for col in entity_cols:
    df_fe = add_entity_basic_features(df_fe, col, top_n=50)

entity_feature_cols = [c for c in df_fe.columns if any(e in c for e in ["_is_null", "_num_tokens", "_freq", "_top50_ou_outros"])]
df_fe[entity_feature_cols].head()

## 9. Features categóricas de negócio

Para as principais variáveis de negócio/jurídico, criamos:

- versão normalizada;
- top N + outros;
- frequência da categoria;
- cardinalidade e cobertura.

Campos:
- carteira;
- produto;
- ação;
- assunto;
- tipo de processo;
- UF;
- comarca;
- fase;
- passível de acordo;
- processo relevante.

In [ ]:
business_cat_cols = [c for c in [
    PORTFOLIO_COL, PRODUCT_COL, ACTION_COL, SUBJECT_COL, PROCESS_TYPE_COL,
    UF_COL, DISTRICT_COL, PROCESS_PHASE_COL, SETTLEMENT_ELIGIBLE_COL,
    RELEVANT_PROCESS_COL
] if c in df_fe.columns]

def add_categorical_business_features(df, col, top_n=30):
    df = df.copy()
    norm_col = f"{col}_norm"
    if norm_col not in df.columns:
        df[norm_col] = df[col].apply(normalize_text)

    freq = df[norm_col].value_counts(dropna=True).to_dict()
    df[f"{col}_freq"] = df[norm_col].map(freq).fillna(0)
    df[f"{col}_freq_log1p"] = np.log1p(df[f"{col}_freq"])

    top_values = set(df[norm_col].value_counts(dropna=True).head(top_n).index)
    df[f"{col}_top{top_n}_ou_outros"] = df[norm_col].apply(lambda x: x if x in top_values else "outros_ou_nulo")

    df[f"{col}_is_null"] = df[norm_col].isna().astype(int)

    return df

for col in business_cat_cols:
    df_fe = add_categorical_business_features(df_fe, col, top_n=30)

business_feature_cols = []
for col in business_cat_cols:
    business_feature_cols += [f"{col}_freq", f"{col}_freq_log1p", f"{col}_top30_ou_outros", f"{col}_is_null"]

df_fe[[c for c in business_feature_cols if c in df_fe.columns]].head()

## 10. Target encoding suavizado exploratório

Criamos taxas históricas suavizadas por grupos relevantes.

Exemplos:
- taxa de perda provável por assunto;
- taxa de perda possível/provável por produto;
- exposição média por carteira;
- valor médio por comarca.

Atenção: para modelo oficial, esse encoding deve ser out-of-fold ou point-in-time. Aqui é EDA/feature engineering exploratório.

In [ ]:
def smooth_target_rate(df, group_col, target_col, k=30, min_count=1):
    tmp = df[[group_col, target_col]].dropna(subset=[target_col]).copy()
    if tmp.empty:
        return None

    global_rate = tmp[target_col].mean()

    agg = (
        tmp.groupby(group_col, dropna=False)[target_col]
        .agg(["sum", "count", "mean"])
        .reset_index()
        .rename(columns={"sum": "target_sum", "count": "qtd_hist", "mean": "target_rate_bruta"})
    )

    agg = agg[agg["qtd_hist"] >= min_count].copy()
    agg[f"{target_col}_rate_suavizada_por_{group_col}"] = (
        (agg["target_sum"] + global_rate * k) / (agg["qtd_hist"] + k)
    )

    return agg

def add_target_encoding_features(df, group_cols, target_cols, k=30, min_count=1):
    df = df.copy()
    reports = {}

    for target_col in target_cols:
        if target_col not in df.columns:
            continue

        for group_col in group_cols:
            if group_col not in df.columns:
                continue

            print(f"Encoding: {target_col} por {group_col}")
            enc = smooth_target_rate(df, group_col, target_col, k=k, min_count=min_count)
            if enc is None:
                continue

            feature_col = f"{target_col}_rate_suavizada_por_{group_col}"
            hist_col = f"qtd_hist_{target_col}_por_{group_col}"

            reports[(target_col, group_col)] = enc.copy()

            df = df.merge(
                enc[[group_col, feature_col, "qtd_hist"]],
                on=group_col,
                how="left"
            )
            df = df.rename(columns={"qtd_hist": hist_col})

            safe_name = re.sub(r"[^a-zA-Z0-9_]", "_", f"{target_col}_by_{group_col}")
            enc.to_csv(REPORTS_DIR / f"target_encoding_{safe_name}.csv", index=False)

    return df, reports

encoding_group_cols = []
for col in [PORTFOLIO_COL, PRODUCT_COL, ACTION_COL, SUBJECT_COL, PROCESS_TYPE_COL, UF_COL, DISTRICT_COL, PROCESS_PHASE_COL, EXTERNAL_LAWYER_COL, INTERNAL_LAWYER_COL, SERVICE_AREA_COL, COURT_UNIT_COL]:
    if col in df_fe.columns:
        # usar versão top/outros quando existir para reduzir ruído
        top_col = f"{col}_top30_ou_outros"
        top50_col = f"{col}_top50_ou_outros"
        if top_col in df_fe.columns:
            encoding_group_cols.append(top_col)
        elif top50_col in df_fe.columns:
            encoding_group_cols.append(top50_col)
        else:
            encoding_group_cols.append(col)

encoding_target_cols = [c for c in ["target_perda_provavel", "target_perda_possivel_ou_provavel", "target_passivel_acordo", "target_processo_relevante"] if c in df_fe.columns]

df_fe, encoding_reports = add_target_encoding_features(
    df_fe,
    group_cols=encoding_group_cols,
    target_cols=encoding_target_cols,
    k=30,
    min_count=1
)

encoding_feature_cols = [c for c in df_fe.columns if "_rate_suavizada_por_" in c or c.startswith("qtd_hist_")]
print("Features de encoding criadas:", len(encoding_feature_cols))
df_fe[encoding_feature_cols].head()

## 11. Features financeiras agregadas por grupo

Além da taxa de perda, criamos estatísticas financeiras por grupo:

- valor médio por assunto;
- valor mediano por produto;
- exposição média por carteira;
- valor p90 por comarca;
- share de valor do grupo.

Essas features ajudam o modelo a entender padrões financeiros por segmento.

In [ ]:
def group_financial_stats(df, group_col, value_col="valor_ajuizado", min_count=1):
    if group_col not in df.columns:
        return None

    total_value = df[value_col].sum()

    agg = (
        df.groupby(group_col, dropna=False)[value_col]
        .agg(["count", "sum", "mean", "median"])
        .reset_index()
        .rename(columns={
            "count": f"qtd_fin_hist_{group_col}",
            "sum": f"valor_total_hist_{group_col}",
            "mean": f"valor_medio_hist_{group_col}",
            "median": f"valor_mediano_hist_{group_col}",
        })
    )

    agg = agg[agg[f"qtd_fin_hist_{group_col}"] >= min_count].copy()
    agg[f"share_valor_hist_{group_col}"] = agg[f"valor_total_hist_{group_col}"] / total_value if total_value else 0

    return agg

financial_group_cols = encoding_group_cols.copy()
financial_stats_reports = {}

for col in financial_group_cols:
    stats = group_financial_stats(df_fe, col, value_col="valor_ajuizado", min_count=1)
    if stats is not None:
        financial_stats_reports[col] = stats
        df_fe = df_fe.merge(stats, on=col, how="left")
        safe_name = re.sub(r"[^a-zA-Z0-9_]", "_", col)
        stats.to_csv(REPORTS_DIR / f"financial_stats_by_{safe_name}.csv", index=False)

financial_stat_cols = [c for c in df_fe.columns if c.startswith(("qtd_fin_hist_", "valor_total_hist_", "valor_medio_hist_", "valor_mediano_hist_", "share_valor_hist_"))]
print("Features financeiras agregadas:", len(financial_stat_cols))
df_fe[financial_stat_cols].head()

## 12. Score heurístico inicial de priorização

Antes de treinar modelo, podemos criar um score de priorização combinando:

- exposição de risco proxy;
- valor ajuizado;
- perda possível/provável;
- passível de acordo;
- processo relevante;
- inatividade;
- fase processual.

Esse score é útil para ranking operacional e validação com o jurídico.

In [ ]:
# Normalização por percentil para combinar variáveis em escalas diferentes
df_fe["rank_exposicao_risco_proxy"] = df_fe["exposicao_risco_proxy"].rank(pct=True)
df_fe["rank_valor_ajuizado"] = df_fe["valor_ajuizado"].rank(pct=True)

components = []

df_fe["score_priorizacao_juridica"] = 0

df_fe["score_priorizacao_juridica"] += 0.35 * df_fe["rank_exposicao_risco_proxy"].fillna(0)
df_fe["score_priorizacao_juridica"] += 0.20 * df_fe["rank_valor_ajuizado"].fillna(0)

if "target_perda_possivel_ou_provavel" in df_fe.columns:
    df_fe["score_priorizacao_juridica"] += 0.15 * df_fe["target_perda_possivel_ou_provavel"].fillna(0)

if "target_passivel_acordo" in df_fe.columns:
    df_fe["score_priorizacao_juridica"] += 0.10 * df_fe["target_passivel_acordo"].fillna(0)

if "target_processo_relevante" in df_fe.columns:
    df_fe["score_priorizacao_juridica"] += 0.10 * df_fe["target_processo_relevante"].fillna(0)

if "flag_sem_andamento_180d" in df_fe.columns:
    df_fe["score_priorizacao_juridica"] += 0.05 * df_fe["flag_sem_andamento_180d"].fillna(0)

if "flag_top_5pct_valor" in df_fe.columns:
    df_fe["score_priorizacao_juridica"] += 0.05 * df_fe["flag_top_5pct_valor"].fillna(0)

df_fe["score_priorizacao_juridica"] = df_fe["score_priorizacao_juridica"].clip(0, 1)

df_fe["prioridade_juridica_categoria"] = pd.cut(
    df_fe["score_priorizacao_juridica"],
    bins=[-0.01, 0.40, 0.70, 1.01],
    labels=["Baixa", "Media", "Alta"]
).astype(str)

df_fe[["score_priorizacao_juridica", "prioridade_juridica_categoria"]].describe(include="all")

In [ ]:
priority_distribution = (
    df_fe.groupby("prioridade_juridica_categoria")
    .agg(
        qtd_processos=(df_fe.columns[0], "count"),
        valor_total=("valor_ajuizado", "sum"),
        exposicao_risco_proxy_total=("exposicao_risco_proxy", "sum"),
        score_medio=("score_priorizacao_juridica", "mean")
    )
    .reset_index()
)
save_report(priority_distribution, "priority_distribution_heuristic_score_benner.csv")
priority_distribution

## 13. Identificar e remover leakage para ABT de modelagem

A remoção depende do objetivo.

## Cenário 1 — Modelo para prever `Estimativa_De_Perda`

Remover:
- `Estimativa_De_Perda`;
- derivados do target;
- `peso_estimativa_perda`;
- `exposicao_risco_proxy`;
- campos de encerramento, se o objetivo for prever antes do desfecho.

## Cenário 2 — Modelo para priorização atual

Pode usar mais campos atuais, mas deve documentar que é score de carteira atual, não previsão histórica point-in-time.

Vamos criar duas ABTs:

1. `abt_benner_score_atual.parquet`
2. `abt_benner_modelo_estimativa_perda_sem_leakage.parquet`

In [ ]:
base_id_cols = existing_cols(df_fe, [TECH_ID_COL, INTERNAL_PROCESS_ID_COL, JUDICIAL_PROCESS_ID_COL])

leakage_common_patterns = [
    "motivo_encerramento",
    "sub_motivo",
    "dataencerramento",
    "estimativa_de_perda",
    "estimativa_perda_score",
    "target_perda",
    "peso_estimativa_perda",
    "exposicao_risco_proxy",
    "score_priorizacao_juridica",
    "prioridade_juridica_categoria",
]

def identify_cols_by_patterns(cols, patterns):
    out = []
    for col in cols:
        c = col.lower()
        if any(p in c for p in patterns):
            out.append(col)
    return sorted(set(out))

leakage_estimativa_perda = identify_cols_by_patterns(df_fe.columns, leakage_common_patterns)

# Preservar IDs e target se necessário em arquivo final separado
pd.DataFrame({"coluna": leakage_estimativa_perda}).to_csv(
    REPORTS_DIR / "leakage_cols_estimativa_perda_benner.csv",
    index=False
)

leakage_estimativa_perda[:100], len(leakage_estimativa_perda)

In [ ]:
# ABT para score atual: mantém campos atuais e score heurístico
abt_score_atual = df_fe.copy()

# ABT para modelo de estimativa de perda: remove leakage do target
abt_modelo_estimativa = df_fe.drop(columns=leakage_estimativa_perda, errors="ignore").copy()

# Reanexar target de modelagem, se existir
if "target_perda_possivel_ou_provavel" in df_fe.columns:
    abt_modelo_estimativa["target_perda_possivel_ou_provavel"] = df_fe["target_perda_possivel_ou_provavel"]

if "target_perda_provavel" in df_fe.columns:
    abt_modelo_estimativa["target_perda_provavel"] = df_fe["target_perda_provavel"]

print("ABT score atual:", abt_score_atual.shape)
print("ABT modelo estimativa:", abt_modelo_estimativa.shape)

## 14. Relatório de tipos de features

Vamos separar numéricas e categóricas e medir cardinalidade para decidir o que entra em modelos baseline.

In [ ]:
def feature_type_report(df, id_cols=None, target_cols=None):
    if id_cols is None:
        id_cols = []
    if target_cols is None:
        target_cols = []

    exclude = set([c for c in id_cols + target_cols if c in df.columns])
    feature_cols = [c for c in df.columns if c not in exclude]

    rows = []
    for col in feature_cols:
        rows.append({
            "feature": col,
            "dtype": str(df[col].dtype),
            "is_numeric": pd.api.types.is_numeric_dtype(df[col]),
            "qtd_distintos": int(df[col].nunique(dropna=True)),
            "perc_nulos": float(df[col].isna().mean()),
            "perc_distintos": float(df[col].nunique(dropna=True) / len(df)) if len(df) else np.nan
        })

    return pd.DataFrame(rows).sort_values(["is_numeric", "qtd_distintos"], ascending=[True, False])

target_cols_estimativa = ["target_perda_possivel_ou_provavel", "target_perda_provavel"]
feature_report_estimativa = feature_type_report(
    abt_modelo_estimativa,
    id_cols=base_id_cols,
    target_cols=target_cols_estimativa
)
save_report(feature_report_estimativa, "feature_type_report_modelo_estimativa_benner.csv")
feature_report_estimativa.head(100)

## 15. Força simples das features numéricas

Mede associação simples com `target_perda_possivel_ou_provavel`.

Não substitui modelagem, mas ajuda a priorizar variáveis.

In [ ]:
def numeric_feature_strength(df, target_col):
    if target_col not in df.columns:
        return pd.DataFrame()

    temp = df[df[target_col].notna()].copy()
    temp[target_col] = temp[target_col].astype(float)

    numeric_cols = [c for c in temp.columns if pd.api.types.is_numeric_dtype(temp[c]) and c != target_col]

    rows = []
    for col in numeric_cols:
        s = pd.to_numeric(temp[col], errors="coerce")
        if s.notna().sum() < 30:
            continue
        try:
            corr = s.corr(temp[target_col])
        except Exception:
            corr = np.nan

        rows.append({
            "feature": col,
            "non_null": int(s.notna().sum()),
            "perc_non_null": float(s.notna().mean()),
            "mean_target_0": s[temp[target_col] == 0].mean(),
            "mean_target_1": s[temp[target_col] == 1].mean(),
            "diff_mean_1_minus_0": s[temp[target_col] == 1].mean() - s[temp[target_col] == 0].mean(),
            "corr_with_target": corr,
            "abs_corr_with_target": abs(corr) if pd.notna(corr) else np.nan
        })

    return pd.DataFrame(rows).sort_values("abs_corr_with_target", ascending=False)

num_strength = numeric_feature_strength(abt_modelo_estimativa, "target_perda_possivel_ou_provavel")
save_report(num_strength, "numeric_feature_strength_estimativa_perda_benner.csv")
num_strength.head(100)

## 16. Força simples das features categóricas

Analisa variação de taxa de perda possível/provável entre categorias.

In [ ]:
def categorical_feature_strength(df, target_col, max_cardinality=100, min_count=30):
    if target_col not in df.columns:
        return pd.DataFrame()

    temp = df[df[target_col].notna()].copy()
    temp[target_col] = temp[target_col].astype(float)

    cat_cols = [
        c for c in temp.columns
        if not pd.api.types.is_numeric_dtype(temp[c])
        and temp[c].nunique(dropna=True) <= max_cardinality
        and c != target_col
    ]

    global_rate = temp[target_col].mean()
    rows = []

    for col in cat_cols:
        agg = (
            temp.groupby(col, dropna=False)[target_col]
            .agg(["count", "mean"])
            .reset_index()
            .rename(columns={"count": "qtd", "mean": "taxa_target"})
        )

        agg = agg[agg["qtd"] >= min_count]
        if agg.empty:
            continue

        agg["abs_diff_vs_global"] = (agg["taxa_target"] - global_rate).abs()

        rows.append({
            "feature": col,
            "qtd_categorias_validas": len(agg),
            "max_taxa_target": agg["taxa_target"].max(),
            "min_taxa_target": agg["taxa_target"].min(),
            "range_taxa_target": agg["taxa_target"].max() - agg["taxa_target"].min(),
            "max_abs_diff_vs_global": agg["abs_diff_vs_global"].max()
        })

    return pd.DataFrame(rows).sort_values("max_abs_diff_vs_global", ascending=False)

cat_strength = categorical_feature_strength(
    abt_modelo_estimativa,
    target_col="target_perda_possivel_ou_provavel",
    max_cardinality=100,
    min_count=30
)
save_report(cat_strength, "categorical_feature_strength_estimativa_perda_benner.csv")
cat_strength.head(100)

## 17. Selecionar features candidatas

Criamos uma lista inicial de features candidatas para modelagem baseline.

In [ ]:
top_numeric = num_strength.head(120)["feature"].tolist() if not num_strength.empty else []
top_categorical = cat_strength.head(80)["feature"].tolist() if not cat_strength.empty else []

candidate_features = list(dict.fromkeys(top_numeric + top_categorical))

candidate_features_df = pd.DataFrame({
    "feature": candidate_features,
    "tipo": ["numeric" if f in top_numeric else "categorical" for f in candidate_features]
})

save_report(candidate_features_df, "candidate_features_estimativa_perda_benner.csv")
candidate_features_df.head(200)

## 18. Criar bases finais

Saídas:

1. `abt_benner_feature_engineering_completa.parquet`
2. `abt_benner_score_atual.parquet`
3. `abt_benner_modelo_estimativa_perda_sem_leakage.parquet`
4. `abt_benner_model_ready_estimativa_perda.parquet`

In [ ]:
# Base completa com todas as features
df_fe.to_parquet(PROCESSED_DIR / "abt_benner_feature_engineering_completa.parquet", index=False)

# Base para score atual
abt_score_atual.to_parquet(PROCESSED_DIR / "abt_benner_score_atual.parquet", index=False)

# Base sem leakage para modelo de estimativa de perda
abt_modelo_estimativa.to_parquet(PROCESSED_DIR / "abt_benner_modelo_estimativa_perda_sem_leakage.parquet", index=False)

# Base reduzida com features candidatas
model_ready_cols = base_id_cols + ["target_perda_possivel_ou_provavel", "target_perda_provavel"] + candidate_features
model_ready_cols = [c for c in model_ready_cols if c in abt_modelo_estimativa.columns]
abt_model_ready = abt_modelo_estimativa[model_ready_cols].copy()

abt_model_ready.to_parquet(PROCESSED_DIR / "abt_benner_model_ready_estimativa_perda.parquet", index=False)

print("Completa:", df_fe.shape)
print("Score atual:", abt_score_atual.shape)
print("Modelo estimativa sem leakage:", abt_modelo_estimativa.shape)
print("Model ready:", abt_model_ready.shape)

## 19. Validar saídas esperadas

In [ ]:
expected_outputs = [
    PROCESSED_DIR / "abt_benner_feature_engineering_completa.parquet",
    PROCESSED_DIR / "abt_benner_score_atual.parquet",
    PROCESSED_DIR / "abt_benner_modelo_estimativa_perda_sem_leakage.parquet",
    PROCESSED_DIR / "abt_benner_model_ready_estimativa_perda.parquet",
    REPORTS_DIR / "target_summary_feature_engineering_benner.csv",
    REPORTS_DIR / "financial_features_report_benner.csv",
    REPORTS_DIR / "priority_distribution_heuristic_score_benner.csv",
    REPORTS_DIR / "leakage_cols_estimativa_perda_benner.csv",
    REPORTS_DIR / "feature_type_report_modelo_estimativa_benner.csv",
    REPORTS_DIR / "numeric_feature_strength_estimativa_perda_benner.csv",
    REPORTS_DIR / "categorical_feature_strength_estimativa_perda_benner.csv",
    REPORTS_DIR / "candidate_features_estimativa_perda_benner.csv",
]

for path in expected_outputs:
    print(path, "OK" if path.exists() else "NÃO ENCONTRADO")

# Roteiro de explicação do Feature Engineering

## 1. Targets possíveis

Criamos targets derivados de `Estimativa_De_Perda`, `Passiveldeacordo` e `Processorelevante`.

## 2. Features financeiras

Transformamos `Valorajuizado` em log, percentis, faixas e flags de alto valor. Isso permite diferenciar causas de massa de causas estratégicas.

## 3. Features temporais

Criamos idade do processo, duração até encerramento e inatividade desde o último andamento.

## 4. Features de contrato

Extraímos presença de contrato, quantidade aproximada de contratos e tempo desde o contrato até o processo.

## 5. Features de entidades

Para autor, escritório, advogado interno e advogado adverso, criamos frequência, top N e indicadores de preenchimento.

## 6. Features categóricas de negócio

Carteira, produto, ação, assunto, tipo de processo, UF, comarca e fase foram normalizados e transformados em frequência/top N.

## 7. Target encoding

Criamos taxas históricas suavizadas por grupos jurídicos. Para produção, isso deve ser refeito de forma out-of-fold ou point-in-time.

## 8. Score heurístico

Criamos um score inicial de priorização jurídica combinando exposição financeira, valor, perda possível/provável, acordo, relevância e inatividade.

## 9. Leakage

Geramos duas ABTs: uma para score atual da carteira e outra mais conservadora para modelo de estimativa de perda.

## 10. Features candidatas

Geramos relatórios de força simples para priorizar features numéricas e categóricas antes do baseline.